# DDR Chart Generator — Training Notebook

**Before running: Runtime → Change runtime type → T4 GPU**

### Assumed setup
Your Google Drive already has StepMania packs uploaded at:
```
MyDrive/ddc_project/data/raw/
  Some Pack Name/
    Song Name/
      Song.sm
      Song.mp3
  Another Pack/
    ...
```
Folder names don't matter. The code searches recursively for any folder
containing both a .sm file and an audio file (.mp3 / .ogg / .wav).

### Run order
**Every session:** re-run cells 1, 2, 3 (Colab VMs are wiped on disconnect).
**First time only:** run cells 4 onwards. Results save to Drive automatically.
**Resuming after disconnect:** re-run cells 1-3, then jump straight to cell 5 with `--curriculum_start` set to the last completed stage.

In [ ]:
# ── 1. Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/ddc_project/data/cache
!mkdir -p /content/drive/MyDrive/ddc_project/data/raw
!mkdir -p /content/drive/MyDrive/ddc_project/checkpoints
!mkdir -p /content/drive/MyDrive/ddc_project/logs
print('Drive mounted.')

In [ ]:
# ── 2. Clone repo and symlink data ──────────────────────────────────────
# Replace with your actual GitHub repo URL
%cd /content
!git clone https://github.com/alexseungum/ieor142b_project.git ddc-chart-gen
%cd /content/ddc-chart-gen

# Symlink Drive data folder so code finds it at 'data/'
!ln -sf /content/drive/MyDrive/ddc_project/data data
print('Repo cloned and data symlinked.')

In [ ]:
# ── 3. Install dependencies ─────────────────────────────────────────────
!pip install librosa soundfile torchaudio --quiet
print('Done.')

In [ ]:
# ── 4. Verify data ───────────────────────────────────────────────────────
import sys
from pathlib import Path
from collections import defaultdict

sys.path.insert(0, '/content/ddc-chart-gen')
from utils.data_utils import parse_chart_file

DATA_ROOT = '/content/drive/MyDrive/ddc_project/data/raw'
root = Path(DATA_ROOT)
audio_exts = {'.ogg', '.mp3', '.wav'}

# ── Find all chart files ──────────────────────────────────────
all_sm  = list(root.rglob('*.sm'))
all_ssc = list(root.rglob('*.ssc'))
song_dirs = sorted({f.parent for f in all_sm + all_ssc})

print(f"{'='*60}")
print(f"DATASET SUMMARY")
print(f"{'='*60}")
print(f"Total song directories found : {len(song_dirs)}")
print(f"  .sm  files                 : {len(all_sm)}")
print(f"  .ssc files                 : {len(all_ssc)}")

# ── Check audio availability ──────────────────────────────────
has_audio  = []
no_audio   = []
for d in song_dirs:
    audio = [f for f in d.iterdir() if f.suffix.lower() in audio_exts]
    if audio:
        has_audio.append(d)
    else:
        no_audio.append(d)

print(f"\nSongs with audio            : {len(has_audio)}")
print(f"Songs missing audio (skipped): {len(no_audio)}")
if no_audio:
    for d in no_audio[:5]:
        print(f"  - {d.relative_to(root)}")
    if len(no_audio) > 5:
        print(f"  ... and {len(no_audio)-5} more")

# ── Per-pack breakdown ────────────────────────────────────────
print(f"\n{'='*60}")
print(f"BREAKDOWN BY PACK")
print(f"{'='*60}")

pack_stats = defaultdict(lambda: {'songs': 0, 'sm': 0, 'ssc': 0, 'no_audio': 0,
                                   'difficulties': defaultdict(int)})

usable_pairs = []
for song_dir in song_dirs:
    pack = song_dir.parent.name
    sm_files    = list(song_dir.glob('*.sm'))
    ssc_files   = list(song_dir.glob('*.ssc'))
    audio_files = [f for f in song_dir.iterdir() if f.suffix.lower() in audio_exts]

    chart_files = sm_files if sm_files else ssc_files
    fmt = 'sm' if sm_files else 'ssc'

    pack_stats[pack]['songs'] += 1
    pack_stats[pack][fmt] += 1

    if not audio_files or not chart_files:
        pack_stats[pack]['no_audio'] += 1
        continue

    usable_pairs.append((str(audio_files[0]), str(chart_files[0]), fmt, pack))

    # Parse chart to get difficulty breakdown
    try:
        sm_data = parse_chart_file(str(chart_files[0]))
        for c in sm_data['charts']:
            if c['chart_type'].lower() in ('dance-single', 'dance single'):
                diff = c['difficulty'].lower()
                pack_stats[pack]['difficulties'][diff] += 1
    except Exception as e:
        pass

diff_order = ['beginner', 'easy', 'medium', 'hard', 'challenge']
for pack, stats in sorted(pack_stats.items()):
    fmt_str = []
    if stats['sm']:  fmt_str.append(f"{stats['sm']} .sm")
    if stats['ssc']: fmt_str.append(f"{stats['ssc']} .ssc")
    no_aud = f"  ⚠ {stats['no_audio']} missing audio" if stats['no_audio'] else ""
    print(f"\n  [{pack}]  {stats['songs']} songs ({', '.join(fmt_str)}){no_aud}")
    diff_counts = [f"{d[:3].upper()}:{stats['difficulties'].get(d,0)}" for d in diff_order]
    print(f"    difficulties: {' | '.join(diff_counts)}")

# ── Overall difficulty breakdown ──────────────────────────────
print(f"\n{'='*60}")
print(f"OVERALL DIFFICULTY BREAKDOWN (usable songs only)")
print(f"{'='*60}")
total_diffs = defaultdict(int)
for _, stats in pack_stats.items():
    for d, n in stats['difficulties'].items():
        total_diffs[d] += n
for d in diff_order:
    print(f"  {d.capitalize():<12}: {total_diffs[d]} charts")

print(f"\nTotal usable song-audio pairs: {len(usable_pairs)}")
print(f"  .sm  : {sum(1 for _,_,f,_ in usable_pairs if f=='sm')}")
print(f"  .ssc : {sum(1 for _,_,f,_ in usable_pairs if f=='ssc')}")

# ── Sanity check one song ─────────────────────────────────────
print(f"\n{'='*60}")
print(f"SANITY CHECK (first usable song)")
print(f"{'='*60}")
if usable_pairs:
    from utils.data_utils import build_sample
    audio_path, chart_path, fmt, pack = usable_pairs[0]
    print(f"Song  : {Path(chart_path).parent.name}  [{fmt.upper()}]")
    print(f"Pack  : {pack}")
    sm_data = parse_chart_file(chart_path)
    print(f"Title : {sm_data['title']}")
    print(f"Charts: {[(c['difficulty'], c['meter']) for c in sm_data['charts'] if 'single' in c['chart_type'].lower()]}")
    sample = build_sample(audio_path, chart_path)
    if sample:
        print(f"X shape      : {sample['X'].shape}  (timesteps, context_window, mel_bins)")
        print(f"y shape      : {sample['y'].shape}  (timesteps, 4 arrows)")
        print(f"Step density : {(sample['y'].sum(-1) > 0).mean():.1%} of timesteps have a step")
    else:
        print("build_sample returned None — no dance-single chart found")
else:
    print("No usable pairs found — check your data folder structure")

In [ ]:
# ── 5. Train ─────────────────────────────────────────────────────────────
# First run: cache builds from scratch (~5-10 min). Subsequent runs load instantly.
# If Colab disconnects mid-training, re-run cells 1-3 then rerun this cell
# with --curriculum_start set to the last stage that printed 'Saved best model'.
#
# epochs_per_stage=5  → quick smoke test (~2 min/stage on T4)
# epochs_per_stage=20 → real training  (~15 min/stage on T4)

import os
DATA_ROOT = '/content/drive/MyDrive/ddc_project/data/raw'
CACHE_DIR = '/content/drive/MyDrive/ddc_project/data/cache'
CKPT_DIR  = '/content/drive/MyDrive/ddc_project/checkpoints'
os.environ['DATA_ROOT'] = DATA_ROOT
os.environ['CACHE_DIR'] = CACHE_DIR
os.environ['CKPT_DIR']  = CKPT_DIR

%cd /content/ddc-chart-gen
!python train.py \
    --data_root      "$DATA_ROOT" \
    --cache_dir      "$CACHE_DIR" \
    --checkpoint_dir "$CKPT_DIR" \
    --epochs_per_stage 20 \
    --batch_size 32 \
    --d_model 256 \
    --n_layers 4 \
    --curriculum_start 0 \
    --num_workers 0

In [ ]:
# ── 5b. Resume after disconnect ──────────────────────────────────────────
# Run this instead of cell 5 if you're resuming after a Colab disconnect.
# First check what stage was last saved:

import torch
CKPT_DIR = '/content/drive/MyDrive/ddc_project/checkpoints'
ckpt = torch.load(f'{CKPT_DIR}/best_model.pt', map_location='cpu')
print(f"Last saved: stage {ckpt['stage']}, epoch {ckpt['epoch']}, val F1 {ckpt['val_f1']:.4f}")
print(f"Set --curriculum_start to {ckpt['stage']} in the train command below")

In [ ]:
# ── 6. Plot loss curves ──────────────────────────────────────────────────
%cd /content/ddc-chart-gen
import json
import matplotlib.pyplot as plt

with open('logs/train_history.json') as f:
    history = json.load(f)

train_h = history['train']
val_h   = history['val']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
stage_colors = {0:'#3498db', 1:'#2ecc71', 2:'#f39c12', 3:'#e74c3c', 4:'#9b59b6'}
stage_names  = {0:'Beginner', 1:'Easy', 2:'Medium', 3:'Hard', 4:'Challenge'}

for metric, ax, title in [
    ('loss',       axes[0], 'Total Loss'),
    ('step_loss',  axes[1], 'Step Placement Loss'),
    ('f1',         axes[2], 'Step F1 Score'),
]:
    for stage in range(5):
        t_vals = [x[metric] for x in train_h if x['stage'] == stage]
        v_vals = [x[metric] for x in val_h   if x['stage'] == stage]
        if not t_vals:
            continue
        color = stage_colors[stage]
        ax.plot(t_vals, color=color, alpha=0.9, label=f'{stage_names[stage]} train')
        ax.plot(v_vals, color=color, alpha=0.5, linestyle='--', label=f'{stage_names[stage]} val')
    ax.set_title(title)
    ax.set_xlabel('Epoch within stage')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

plt.suptitle('Training curves by curriculum stage', y=1.02)
plt.tight_layout()
plt.savefig('logs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
!mkdir -p /content/drive/MyDrive/ddc_project/logs
!cp logs/training_curves.png /content/drive/MyDrive/ddc_project/logs/
print('Saved to Drive.')

In [ ]:
# ── 7. Generate a chart ──────────────────────────────────────────────────
# Option A: use a song already in your dataset
# Option B: upload any song from your computer

%cd /content/ddc-chart-gen
import os
from pathlib import Path

# Option A: pick a song from the dataset
# DATA_ROOT = '/content/drive/MyDrive/ddc_project/data/raw'
# all_sm = list(Path(DATA_ROOT).rglob('*.sm'))
# for sm in all_sm:
#     audio = [f for f in sm.parent.iterdir()
#              if f.suffix.lower() in {'.ogg', '.mp3', '.wav'}]
#     if audio:
#         test_audio = str(audio[0])
#         print(f'Using: {sm.parent.name} — {audio[0].name}')
#         break

# Option B: upload any song from your computer (mp3, ogg, wav)
from google.colab import files as colab_files
uploaded = colab_files.upload()
test_audio = list(uploaded.keys())[0]
print(f'Using: {test_audio}')

CKPT_DIR = '/content/drive/MyDrive/ddc_project/checkpoints'
os.environ['TEST_AUDIO'] = test_audio
os.environ['CKPT_DIR']   = CKPT_DIR

# Adjust --difficulty (0-4) and --threshold (0.1-0.9, lower = more arrows)
!python generate.py \
    --audio      "$TEST_AUDIO" \
    --checkpoint "$CKPT_DIR/best_model.pt" \
    --difficulty 2 \
    --threshold  0.5 \
    --output     output_chart

!cp -r output_chart /content/drive/MyDrive/ddc_project/
print('Done! Run cell 9 to download visualizer.html')

In [ ]:
# ── 8. Threshold sweep ───────────────────────────────────────────────────
# Shows how the step threshold knob controls chart density.
# Lower threshold = more arrows = harder feel.
# Higher threshold = fewer arrows = easier feel.

%cd /content/ddc-chart-gen
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '/content/ddc-chart-gen')

from models.model import DDRTransformer, generate_chart
from generate import audio_to_model_input

CKPT_DIR = '/content/drive/MyDrive/ddc_project/checkpoints'
ckpt = torch.load(f'{CKPT_DIR}/best_model.pt', map_location='cpu')
model_args = ckpt.get('args', {})
model = DDRTransformer(
    d_model=model_args.get('d_model', 256),
    nhead=model_args.get('nhead', 8),
    num_encoder_layers=model_args.get('n_layers', 4),
    dim_feedforward=model_args.get('d_ff', 1024),
    dropout=0.0,
)
model.load_state_dict(ckpt['model_state'])
X = audio_to_model_input(test_audio)

thresholds = np.linspace(0.1, 0.9, 17)
densities  = []
for th in thresholds:
    mask, _ = generate_chart(model, X, difficulty=2, step_threshold=float(th))
    densities.append(mask.mean())

plt.figure(figsize=(7, 4))
plt.plot(thresholds, densities, 'o-', color='#e74c3c')
plt.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='default threshold')
plt.xlabel('Step threshold')
plt.ylabel('Fraction of timesteps with a step')
plt.title('Difficulty knob: threshold vs. chart density')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('logs/threshold_sweep.png', dpi=150)
plt.show()
!mkdir -p /content/drive/MyDrive/ddc_project/logs
!cp logs/threshold_sweep.png /content/drive/MyDrive/ddc_project/logs/

In [ ]:
# ── 9. Download results ──────────────────────────────────────────────────
# Everything is also on Drive already. This downloads directly to your computer.
%cd /content/ddc-chart-gen
CKPT_DIR = '/content/drive/MyDrive/ddc_project/checkpoints'

from google.colab import files
files.download('output_chart/visualizer.html')   # open in browser to see the chart
files.download('output_chart/chart.sm')          # load in StepMania
files.download(f'{CKPT_DIR}/best_model.pt')      # trained model weights
files.download('logs/training_curves.png')       # loss curves
files.download('logs/threshold_sweep.png')       # difficulty knob plot